# 02 — Nettoyage des données

**Objectif** : produire `data/processed/ventes_plage_prix_propre.csv`.

La logique vit dans `src/data_cleaning.py`. Ce notebook l'applique et vérifie
que les totaux recoupent la publication officielle de quebec.ca.

In [4]:
import importlib
import sys
from pathlib import Path

racine = Path.cwd().resolve()
if not (racine / "src").is_dir():
    racine = racine.parent
sys.path.insert(0, str(racine))

import src
import src.data_cleaning
import src.referentiels

importlib.reload(src)
importlib.reload(src.data_cleaning)
importlib.reload(src.referentiels)

import pandas as pd

from src import VENTES_BRUTES_CSV, VENTES_PROPRES_CSV
from src.data_cleaning import nettoyer_ventes
from src.referentiels import CLES_UNICITE, REGION_ABSENTE

pd.set_option("display.max_columns", None)

In [5]:
brut = pd.read_csv(VENTES_BRUTES_CSV, dtype=str, encoding="utf-8-sig")
propre = nettoyer_ventes(brut)

print(f"{len(brut)} lignes en entrée → {len(propre)} lignes après nettoyage")
propre.dtypes

3262 lignes en entrée → 3262 lignes après nettoyage


periode             datetime64[us]
id_region                   string
code_plage_prix             string
code_nature_acte            string
nombre_ventes                Int64
region                         str
plage_prix                     str
nature_acte                    str
dtype: object

In [3]:
propre.head(8)

,periode,id_region,code_plage_prix,code_nature_acte,nombre_ventes,region,plage_prix,nature_acte
0,2021-01-01,01,1,001,240,Bas-Saint-Laurent,Moins de 250 000 $,Vente
1,2021-01-01,01,2,001,29,Bas-Saint-Laurent,De 250 000 $ à 500 000 $,Vente
2,2021-01-01,01,3,001,6,Bas-Saint-Laurent,Plus de 500 000 $,Vente
3,2021-01-01,02,1,001,298,Saguenay–Lac-Saint-Jean,Moins de 250 000 $,Vente
4,2021-01-01,02,2,001,48,Saguenay–Lac-Saint-Jean,De 250 000 $ à 500 000 $,Vente
5,2021-01-01,02,3,001,16,Saguenay–Lac-Saint-Jean,Plus de 500 000 $,Vente
6,2021-01-01,03,1,001,497,Capitale-Nationale,Moins de 250 000 $,Vente
7,2021-01-01,03,2,001,295,Capitale-Nationale,De 250 000 $ à 500 000 $,Vente


## Contrôles de qualité

On échoue explicitement si une hypothèse structurante est violée.

In [6]:
assert not propre.empty, "Le jeu nettoyé est vide."
assert not propre.duplicated(subset=list(CLES_UNICITE)).any(), "Doublons résiduels."
assert propre["periode"].notna().all(), "Périodes non converties."
assert (propre["nombre_ventes"] >= 0).all(), "Ventes négatives."
assert REGION_ABSENTE not in set(propre["id_region"]), "La région 10 ne devrait pas apparaître."
assert propre["region"].notna().all(), "Région sans libellé."
assert propre["plage_prix"].notna().all(), "Plage de prix sans libellé."

print(f"Période : {propre['periode'].min():%Y-%m} → {propre['periode'].max():%Y-%m}")
print(f"Régions : {propre['id_region'].nunique()} | Plages : {propre['code_plage_prix'].nunique()}")
propre.describe(include="all").T

Période : 2021-01 → 2026-08
Régions : 16 | Plages : 3


,count,unique,top,freq,mean,min,25%,50%,75%,max,std
periode,3262,NaN,NaN,NaN,2023-10-17 04:13:23.433476,2021-01-01 00:00:00,2022-06-01 00:00:00,2023-11-01 00:00:00,2025-03-24 06:00:00,2026-08-01 00:00:00,NaN
id_region,3262,16,01,204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
code_plage_prix,3262,3,1,1088,NaN,NaN,NaN,NaN,NaN,NaN,NaN
code_nature_acte,3262,1,001,3262,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nombre_ventes,3262.0,<NA>,<NA>,<NA>,321.961067,1.0,84.0,214.0,419.0,4100.0,357.434556
region,3262,16,Bas-Saint-Laurent,204,NaN,NaN,NaN,NaN,NaN,NaN,NaN
plage_prix,3262,3,Moins de 250 000 $,1088,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nature_acte,3262,1,Vente,3262,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Récolement avec quebec.ca (août 2026)

Les totaux officiels publiés le 15 septembre 2026 servent de test de non-régression
sur le décodage des plages de prix.

In [7]:
OFFICIEL_AOUT_2026 = {
    "Moins de 250 000 $": 2799,
    "De 250 000 $ à 500 000 $": 4496,
    "Plus de 500 000 $": 4971,
}

aout = propre.loc[propre["periode"] == "2026-08-01"]
observé = aout.groupby("plage_prix")["nombre_ventes"].sum().to_dict()

print(f"Total août 2026 : {aout['nombre_ventes'].sum():,} (officiel : 12 266)")
for libelle, attendu in OFFICIEL_AOUT_2026.items():
    print(f"  {libelle}: {observé[libelle]:,} (officiel : {attendu:,})")
    assert observé[libelle] == attendu, f"Écart sur {libelle}"

assert aout["nombre_ventes"].sum() == 12266

Total août 2026 : 12,266 (officiel : 12 266)
  Moins de 250 000 $: 2,799 (officiel : 2,799)
  De 250 000 $ à 500 000 $: 4,496 (officiel : 4,496)
  Plus de 500 000 $: 4,971 (officiel : 4,971)


## Sauvegarde

In [8]:
VENTES_PROPRES_CSV.parent.mkdir(parents=True, exist_ok=True)
propre.to_csv(VENTES_PROPRES_CSV, index=False, encoding="utf-8")
print(f"Écrit : {VENTES_PROPRES_CSV}")

Écrit : C:\Users\Claude1\Documents\Projet d'analyse\Projet-marche-immobilier-quebec\data\processed\ventes_plage_prix_propre.csv
